# Online Mode (End-to-End Managed) — Speculative Decoding Training

This notebook demonstrates the **ONLINE** training mode of `SpeculativeDecodingTrainer`
from the Kubeflow SDK on Red Hat OpenShift AI. ONLINE is the simplest workflow — the
SDK manages everything in a single job:

1. **Deploys a vLLM sidecar** to serve the verifier model
2. **Extracts hidden states** from the dataset batch by batch
3. **Trains the Eagle3 draft model** using the extracted hidden states

Hidden states are processed in a streaming fashion — each batch is extracted, used for
training, then discarded. This means disk usage stays constant regardless of dataset
size, unlike `DATA_ONLY` which saves all hidden states to disk.

## Background

Large language models generate tokens one at a time, and each token requires reading the
entire model from GPU memory — making inference **memory-bound**. Speculative decoding
exploits this: a small, fast **draft model** guesses the next several tokens, then the
large **verifier model** checks all guesses in a single forward pass. The output is
mathematically identical to normal decoding — no quality loss.

**Eagle3** is a draft model architecture that reads hidden states from four intermediate
layers of the verifier (not just the final logits), giving it richer context for more
accurate predictions.

This notebook uses the `magpie` built-in dataset.

## Hardware Requirements

The table below shows the **minimum** resources needed. See the Configuration cell for
recommended values that improve performance.

| Component | GPU (min) | GPU (recommended) | CPU (min) | CPU (rec.) | Memory (min) | Memory (rec.) |
|-----------|-----------|-------------------|-----------|------------|-------------|--------------|
| Training container | 1× GPU | 2× GPU | 1 core | 4 cores | 32Gi | 64Gi |
| vLLM sidecar | 1× GPU | 1× GPU | 1 core | 4 cores | 48Gi | 96Gi |
| **Total** | **2 GPUs** | **3 GPUs** | **2 cores** | **8 cores** | **80Gi** | **160Gi** |

**Trade-off:** ONLINE is the simplest (one step instead of two), but you cannot reuse
the extracted data for multiple training runs with different hyperparameters. If you want
to experiment with hyperparameters, use
[DATA_ONLY](../data-only/) + [TRAIN_ONLY](../train-only/) instead.

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,  # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,  # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,  # Enum: EAGLE3 (currently the only supported type)
)

# Kubernetes Python client — used to configure API server auth
from kubernetes import client as k8s

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

The following environment variables are required for API authentication:

- `OPENSHIFT_API_URL` — your cluster API URL (e.g., `https://api.cluster.example.com:6443`)
- `NOTEBOOK_USER_TOKEN` — an access token for API calls

In OpenShift AI workbenches, these are often auto-set.

If they are not set in your environment, uncomment and populate the values in the next cell.

In [ ]:
# ============================================================================
# AUTHENTICATION
# ============================================================================
# If your workbench does not auto-populate these env vars, uncomment and fill them in:
#
# api_server = "https://api.your-cluster.example.com:6443"
# token = "sha256~your-token-here"

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")

if not api_server or not token:
    raise RuntimeError(
        "OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set. "
        "Either set them in your environment or uncomment the values above."
    )

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# Configure Kubernetes client
configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False  # Set to True if using trusted certificates
configuration.api_key = {"authorization": f"Bearer {token}"}

# ============================================================================
# PVC MOUNT PATHS
# ============================================================================
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        "Expected workbench PVC mount not found at: "
        f"{NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name/mount, update PVC_NAME/NOTEBOOK_SHARED_PATH.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# TRAINER CLIENT
# ============================================================================
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(client_configuration=configuration)
)

# ClusterTrainingRuntime (CTR) for ONLINE mode.
# This CTR includes both a vLLM sidecar and training container in one pod.
ONLINE_CTR = "vllm-extract-cuda"

# Verify the CTR exists on the cluster
available_runtimes = {r.name for r in trainer_client.list_runtimes()}
status = (
    "Found" if ONLINE_CTR in available_runtimes else "WARNING: not found on cluster"
)
print(f"CTR '{ONLINE_CTR}': {status}")

print(f"\nAPI Server: {api_server}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## (Optional) Download the Verifier Model

Pre-downloading the verifier model to the shared PVC speeds up pod startup — the
training pods will find the model on the PVC instead of downloading it from
HuggingFace during the job.

Skip this cell if the model is already on your PVC or if you prefer to let the
training pods download it automatically (requires `HF_TOKEN` in the `env` parameter).

In [ ]:
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = HF_TOKEN

model_id = "Qwen/Qwen3-0.6B"
local_dir = f"{NOTEBOOK_SHARED_PATH}/models/Qwen3-0.6B"

snapshot_download(model_id, local_dir=local_dir)
print(f"Model downloaded to {local_dir}")

## Configuration

The following constants configure the training run. The verifier model is
[Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), a 28-layer transformer,
specified by its HuggingFace model ID. The training pods download the model
automatically — no manual pre-download is required (`HF_TOKEN` provides
authentication).

All output paths use **PVC URIs** (`pvc://<pvc-name>/<path>`), which the SDK
resolves to container mount paths internally.

`RUN_NAME` creates a namespace on the PVC for each experiment — change it to start
a fresh run without overwriting previous results.

In [ ]:
# Unique run identifier — namespaces all output paths on the PVC.
# Change this to start a fresh experiment without overwriting previous results.
RUN_NAME = "run-01"

# Set the Verifier Model for the training job.
VERIFIER_MODEL = "Qwen/Qwen3-0.6B"
# If you pre-downloaded the model to the PVC, use the PVC URI instead:
# VERIFIER_MODEL = f"pvc://{PVC_NAME}/models/Qwen3-0.6B"

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-0.6B has 28 transformer layers (indexed 1-28).
# Layers chosen: early (2), mid (14), late (25), and final (28) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
TARGET_LAYER_IDS = [2, 14, 25, 28]

# Minimum resources for the training container.
# 1 GPU is sufficient to train the small Eagle3 draft model (~1.2 GB with Qwen3-0.6B).
TRAINING_RESOURCES = {
    "nvidia.com/gpu": 1,  # Recommended: 2 — enables data-parallel training
    "cpu": "1",  # Recommended: "4" — faster data loading and preprocessing
    "memory": "32Gi",  # Recommended: "64Gi" — more headroom for optimizer state
}

# Minimum resources for the vLLM sidecar that serves the verifier during extraction.
# The sidecar is hard-limited to exactly 1 GPU — more raises a ValueError.
VLLM_RESOURCES = {
    "nvidia.com/gpu": 1,
    "cpu": "1",  # Recommended: "4" — faster tokenization and model loading
    "memory": "48Gi",  # Recommended: "96Gi" — more headroom for KV cache
}

# Training hyperparameters
EPOCHS = 3  # Number of full passes over the training data
LEARNING_RATE = 1e-4  # AdamW learning rate — 1e-4 is a good starting point for Eagle3
TOTAL_SEQ_LEN = 2048  # Maximum sequence length for both extraction and training
MAX_SAMPLES = 500  # Cap on the number of dataset samples to process

print("Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  Training GPUs:     {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  vLLM GPUs:         {VLLM_RESOURCES['nvidia.com/gpu']}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Learning rate:     {LEARNING_RATE}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")
print(f"  Max samples:       {MAX_SAMPLES}")

## Online Mode (End-to-End Managed)

The `ONLINE` mode is the simplest workflow — the SDK manages everything in a single job:

1. **Deploys a vLLM sidecar** to serve the verifier model
2. **Extracts hidden states** from the dataset batch by batch
3. **Trains the Eagle3 draft model** using the extracted hidden states

Hidden states are processed in a streaming fashion — each batch is extracted, used for
training, then discarded. This means disk usage stays constant regardless of dataset
size, unlike `DATA_ONLY` which saves all hidden states to disk.

**Trade-off:** ONLINE is simpler (one step instead of two), but you cannot reuse the
extracted data for multiple training runs with different hyperparameters. If you want to
experiment with hyperparameters, use `DATA_ONLY` + `TRAIN_ONLY` instead.

We use the `magpie` built-in dataset for this example.

In [ ]:
ONLINE_JOB = f"eagle3-online-{RUN_NAME}"
ONLINE_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}-online"

# Configure the ONLINE trainer — the simplest end-to-end workflow.
# The SDK manages everything: deploys vLLM sidecar, extracts hidden states, and trains.
# Hidden states are streamed batch-by-batch and discarded after use, so disk usage
# stays constant regardless of dataset size (unlike DATA_ONLY which persists all states).
online_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.ONLINE,  # Fully managed: extraction + training in one step
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,
    dataset_name="magpie",
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    vllm_resources=VLLM_RESOURCES,  # Resources for the managed vLLM sidecar
    vllm_gpu_memory_utilization=0.9,
    training_resources=TRAINING_RESOURCES,  # Resources for the training container
    regenerate_responses=True,  # Generate fresh responses from prompts (not reuse dataset answers)
    enable_progression_tracking=True,  # Enable SDK-side progress polling
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=ONLINE_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
    ),
    env={"HF_TOKEN": HF_TOKEN},
)

print("ONLINE Configuration:")
print(f"  Job name:      {ONLINE_JOB}")
print(f"  Mode:          {online_trainer.mode.value}")
print(f"  Verifier:      {online_trainer.verifier_model}")
print(f"  Dataset:       {online_trainer.dataset_name}")
print(f"  Max samples:   {online_trainer.max_samples}")
print(f"  Target layers: {online_trainer.config.target_layer_ids}")
print(f"  Training GPUs: {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  vLLM GPUs:     {VLLM_RESOURCES['nvidia.com/gpu']}")
print(f"  Output dir:    {online_trainer.output_dir}")

In [ ]:
# Submit the ONLINE TrainJob to the cluster.
# Uses the ONLINE_CTR which includes both a vLLM sidecar and training container.
trainer_client.train(
    options=[Name(name=ONLINE_JOB)],
    trainer=online_trainer,
    runtime=ONLINE_CTR,
)

print(f"ONLINE job submitted: {ONLINE_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={ONLINE_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the ONLINE job (Pending, Running, Succeeded, Failed).
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(ONLINE_JOB)

## Cleanup

Delete the TrainJob when you are done.

In [ ]:
# Delete the completed TrainJob to free cluster resources (pods, volumes, etc.).
# Note: Deleting a job does NOT delete the output data on the PVC —
# the trained draft model checkpoint remains available for deployment.

# trainer_client.delete_job(ONLINE_JOB)

# print("TrainJob deleted.")

## Summary

This notebook demonstrated fully managed end-to-end Eagle3 draft model training using
the **ONLINE** mode of `SpeculativeDecodingTrainer`. The SDK handled vLLM deployment,
hidden state extraction, and draft model training — all in a single job.

### Key Takeaways

- ONLINE mode is the **simplest path** — one job, one command, no external dependencies.
- Hidden states are **streamed and discarded**, keeping disk usage constant regardless of
  dataset size.
- The trade-off: you **cannot reuse** extracted data for multiple training runs with
  different hyperparameters.
- All storage paths use **PVC URIs** (`pvc://<pvc-name>/<path>`) — the SDK handles
  volume mounting internally.

### Next Steps

- **Deploy the trained draft model** with vLLM for speculative decoding inference.
- Adjust `EPOCHS`, `LEARNING_RATE`, and `MAX_SAMPLES` to tune draft model quality.
- If you want to **experiment with hyperparameters** without re-running extraction,
  use [DATA_ONLY](../data-only/) + [TRAIN_ONLY](../train-only/) instead.
- If you already have a **running vLLM deployment**, try [OFFLINE mode](../offline/)
  to reuse it for extraction.